# Day 1: GitHub Repository Data Ingestion

This notebook implements a system to download GitHub repositories and extract structured documentation. This is the foundation for building a RAG (Retrieval Augmented Generation) system that can answer questions about code repositories.

## What We're Building

A function that:
1. Downloads a GitHub repository as a zip archive
2. Extracts markdown documentation files (.md and .mdx)
3. Parses YAML frontmatter metadata from each file
4. Returns structured data ready for indexing

## Key Concepts

**Frontmatter**: YAML metadata block at the start of markdown files, commonly used in Jekyll, Hugo, and Next.js documentation sites. Format:
```yaml
---
title: Getting Started
description: Introduction guide
sidebar_position: 1
---

# Markdown content here
```

**In-Memory Processing**: We process the zip archive directly in memory (using BytesIO) rather than writing to disk, which is more efficient for our use case.

In [ ]:
# Import required libraries
import requests
from zipfile import ZipFile
from io import BytesIO, StringIO
import frontmatter
from pathlib import Path
from typing import List, Dict, Any

## Helper Functions

First, we'll create helper functions to handle specific parts of the workflow:
1. Downloading the repository as a zip archive
2. Filtering for markdown files only
3. Parsing frontmatter from markdown content

In [ ]:
def download_repo_zip(repo_owner: str, repo_name: str) -> bytes:
    """Download GitHub repository as zip archive.

    Uses GitHub's codeload API which provides repositories as zip archives
    without requiring authentication for public repos.

    Args:
        repo_owner: GitHub username or organization (e.g., 'DataTalksClub')
        repo_name: Repository name (e.g., 'faq')

    Returns:
        Binary content of zip archive

    Raises:
        requests.HTTPError: If download fails (repo not found, network error, etc.)
    """
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    print(f"Downloading {repo_owner}/{repo_name} from {url}")

    response = requests.get(url)
    response.raise_for_status()  # Raise exception for HTTP errors

    print(f"Downloaded {len(response.content)} bytes")
    return response.content

In [ ]:
def is_markdown_file(filename: str) -> bool:
    """Check if file is a markdown document (.md or .mdx).

    Args:
        filename: Full path to file in zip archive

    Returns:
        True if file ends with .md or .mdx
    """
    path = Path(filename)
    return path.suffix in ['.md', '.mdx']

In [ ]:
def parse_markdown_with_frontmatter(content_bytes: bytes) -> Dict[str, Any]:
    """Parse markdown file extracting frontmatter metadata and content.

    Frontmatter is YAML metadata between --- markers at file start:
    ---
    title: Page Title
    description: Page description
    ---
    # Markdown content

    Args:
        content_bytes: Raw file content as bytes

    Returns:
        Dictionary with 'metadata' (frontmatter dict) and 'content' (markdown string)
    """
    # Decode bytes to string (GitHub files are UTF-8)
    content_str = content_bytes.decode('utf-8', errors='ignore')

    # Parse using frontmatter library
    post = frontmatter.loads(content_str)

    return {
        'metadata': dict(post.metadata),  # Convert to plain dict
        'content': post.content  # Markdown content after frontmatter
    }

## Main Function: read_repo_data()

Now we combine the helpers into the main function that orchestrates the entire workflow:
1. Download repo as zip
2. Open zip in memory (no disk I/O)
3. Filter for markdown files only
4. Parse each markdown file's frontmatter and content
5. Return structured list of documents